In [6]:
import numpy as np
from scipy.signal import butter, filtfilt, welch

def bandpass(x, low=4, high=45, fs=128, order=4):
    b, a = butter(order, [low, high], btype="band", fs=fs)
    return filtfilt(b, a, x, axis=-1)

def preprocess_subject(d, win=512, step=256):
    data = d["data"][:, :32, 384:]
    data = bandpass(data)
    mean = data.mean(axis=(0, 2), keepdims=True)
    std = data.std(axis=(0, 2), keepdims=True)
    data = ((data - mean) / std).astype(np.float32)

    y_trial = (d["labels"][:, :2] > 5).astype(int)

    X, y, trial_id = [], [], []
    for t in range(data.shape[0]):
        for start in range(0, data.shape[2] - win + 1, step):
            X.append(data[t, :, start:start + win])
            y.append(y_trial[t])
            trial_id.append(t)
    return np.array(X), np.array(y), np.array(trial_id)

def band_power_features(X, fs=128):
    bands = {"theta": (4, 8), "alpha": (8, 13), "beta": (13, 30), "gamma": (30, 45)}
    f, psd = welch(X, fs=fs, nperseg=128, axis=-1)      # (n, 32, freqs)
    feats = []
    for lo, hi in bands.values():
        idx = (f >= lo) & (f < hi)
        feats.append(np.log(psd[:, :, idx].mean(axis=-1) + 1e-10))
    return np.concatenate(feats, axis=1)                # (n, 32*4 = 128)

In [7]:
import numpy as np, os
from load_data import load_subject
from preprocess import preprocess_subject

os.makedirs("data/processed", exist_ok=True)
for sid in range(1, 33):
    d = load_subject(sid)
    X, y, tid = preprocess_subject(d)
    np.savez_compressed(f"data/processed/s{sid:02d}.npz", X=X, y=y, trial_id=tid)
    print(sid, X.shape)

NameError: name 'null' is not defined

In [8]:
code = '''import numpy as np
from scipy.signal import butter, filtfilt, welch

def bandpass(x, low=4, high=45, fs=128, order=4):
    b, a = butter(order, [low, high], btype="band", fs=fs)
    return filtfilt(b, a, x, axis=-1)

def preprocess_subject(d, win=512, step=256):
    data = d["data"][:, :32, 384:]
    data = bandpass(data)
    mean = data.mean(axis=(0, 2), keepdims=True)
    std = data.std(axis=(0, 2), keepdims=True)
    data = ((data - mean) / std).astype(np.float32)

    y_trial = (d["labels"][:, :2] > 5).astype(int)

    X, y, trial_id = [], [], []
    for t in range(data.shape[0]):
        for start in range(0, data.shape[2] - win + 1, step):
            X.append(data[t, :, start:start + win])
            y.append(y_trial[t])
            trial_id.append(t)
    return np.array(X), np.array(y), np.array(trial_id)

def band_power_features(X, fs=128):
    bands = {"theta": (4, 8), "alpha": (8, 13), "beta": (13, 30), "gamma": (30, 45)}
    f, psd = welch(X, fs=fs, nperseg=128, axis=-1)
    feats = []
    for lo, hi in bands.values():
        idx = (f >= lo) & (f < hi)
        feats.append(np.log(psd[:, :, idx].mean(axis=-1) + 1e-10))
    return np.concatenate(feats, axis=1)
'''
with open("preprocess.py", "w") as f:
    f.write(code)
print(open("preprocess.py").read()[:200])

import numpy as np
from scipy.signal import butter, filtfilt, welch

def bandpass(x, low=4, high=45, fs=128, order=4):
    b, a = butter(order, [low, high], btype="band", fs=fs)
    return filtfilt(b,


In [9]:
import numpy as np, os
from load_data import load_subject
from preprocess import preprocess_subject

os.makedirs("data/processed", exist_ok=True)
for sid in range(1, 33):
    d = load_subject(sid)
    X, y, tid = preprocess_subject(d)
    np.savez_compressed(f"data/processed/s{sid:02d}.npz", X=X, y=y, trial_id=tid)
    print(sid, X.shape)

C:\Users\ishap\Downloads\AI for Neuromorphic emotion behavior modeling\load_data.py:5: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  "execution_count": null,


1 (1160, 32, 512)
2 (1160, 32, 512)
3 (1160, 32, 512)
4 (1160, 32, 512)
5 (1160, 32, 512)
6 (1160, 32, 512)
7 (1160, 32, 512)
8 (1160, 32, 512)
9 (1160, 32, 512)
10 (1160, 32, 512)
11 (1160, 32, 512)
12 (1160, 32, 512)
13 (1160, 32, 512)
14 (1160, 32, 512)
15 (1160, 32, 512)
16 (1160, 32, 512)
17 (1160, 32, 512)
18 (1160, 32, 512)
19 (1160, 32, 512)
20 (1160, 32, 512)
21 (1160, 32, 512)
22 (1160, 32, 512)
23 (1160, 32, 512)
24 (1160, 32, 512)
25 (1160, 32, 512)
26 (1160, 32, 512)
27 (1160, 32, 512)
28 (1160, 32, 512)
29 (1160, 32, 512)
30 (1160, 32, 512)
31 (1160, 32, 512)
32 (1160, 32, 512)


In [11]:
from preprocess import band_power_features
from sklearn.svm import SVC
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupKFold, cross_val_score

# start with one subject
z = np.load("data/processed/s01.npz")
X, y, tid = z["X"], z["y"], z["trial_id"]
F = band_power_features(X)
print(F.shape)                                   # expect (1160, 128)

clf = make_pipeline(StandardScaler(), SVC(kernel="rbf"))
cv = GroupKFold(n_splits=5)

for name, col in [("valence", 0), ("arousal", 1)]:
    scores = cross_val_score(clf, F, y[:, col], groups=tid, cv=cv)
    print(name, "accuracy:", scores.mean().round(3))

(1160, 128)
valence accuracy: 0.491
arousal accuracy: 0.603


In [12]:
import numpy as np
from preprocess import band_power_features
from sklearn.svm import SVC
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupKFold, cross_val_score

results = {"valence": [], "arousal": []}
for sid in range(1, 33):
    z = np.load(f"data/processed/s{sid:02d}.npz")
    F = band_power_features(z["X"])
    clf = make_pipeline(StandardScaler(), SVC(kernel="rbf"))
    for name, col in [("valence", 0), ("arousal", 1)]:
        acc = cross_val_score(clf, F, z["y"][:, col], groups=z["trial_id"],
                              cv=GroupKFold(n_splits=5)).mean()
        results[name].append(acc)
    print(sid, round(results["valence"][-1], 3), round(results["arousal"][-1], 3))

print("MEAN valence:", np.mean(results["valence"]).round(3))
print("MEAN arousal:", np.mean(results["arousal"]).round(3))

1 0.491 0.603
2 0.628 0.564
3 0.664 0.758
4 0.666 0.625
5 0.603 0.49
6 0.694 0.535
7 0.593 0.537
8 0.51 0.549
9 0.655 0.559
10 0.741 0.582
11 0.493 0.661
12 0.561 0.825
13 0.778 0.831
14 0.709 0.638
15 0.73 0.509
16 0.643 0.63
17 0.513 0.653
18 0.706 0.526
19 0.592 0.603
20 0.711 0.759
21 0.474 0.768
22 0.698 0.514
23 0.656 0.734
24 0.583 0.825
25 0.561 0.697
26 0.624 0.58
27 0.688 0.607
28 0.603 0.489
29 0.618 0.6
30 0.638 0.534
31 0.564 0.592
32 0.465 0.594
MEAN valence: 0.62
MEAN arousal: 0.624


In [2]:
import numpy as np, torch
from preprocess import band_power_features
from sklearn.model_selection import GroupShuffleSplit

z = np.load("data/processed/s01.npz")
F = band_power_features(z["X"]).astype(np.float32)
y = z["y"][:, 0]                      # 0 = valence, 1 = arousal
tid = z["trial_id"]

tr, te = next(GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=0).split(F, y, tid))

mu, sd = F[tr].mean(0), F[tr].std(0) + 1e-8        # statistics from TRAIN only
def scale(a):                                       # map to [0, 1] for rate coding
    return np.clip((a - mu) / sd, -3, 3) / 6 + 0.5

Xtr = torch.tensor(scale(F[tr]), dtype=torch.float32)
Xte = torch.tensor(scale(F[te]), dtype=torch.float32)
ytr = torch.tensor(y[tr]); yte = torch.tensor(y[te])
print(Xtr.shape, Xte.shape)

torch.Size([870, 128]) torch.Size([290, 128])
